# SciCite Inline Word-Level ToW Dataset Generator

This notebook generates the **Inline Word-Level ToW (IW-ToW)** dataset used in the SciCite extension experiments.

IW-ToW follows the original Thoughts of Words setup at the word level. For each citation context, the notebook selects a non-trivial target word, shows the API model only the prefix before that word, asks it to predict and explain the likely next word, checks whether the explanation is consistent with the observed target word, and inserts a short `<ToW>...</ToW>` annotation immediately before the target word.

## Inputs

The notebook expects raw SciCite sample files in Google Drive:

- `scicite_train_raw_sample.jsonl`
- `scicite_validation_raw_sample.jsonl`
- `scicite_test_raw_sample.jsonl`

## Outputs

The cleaned IW-ToW files are written to:

`/content/drive/MyDrive/tow_project/B/extension/scicite/wordlevel_tow_generations_clean_v2/`

This notebook only generates the IW-ToW dataset. It does not train or evaluate classifiers.


## 1. Install dependencies


In [ ]:
!pip install -q "openai>=1.0.0" "pandas==2.2.2"

## 2. Mount Drive and configure paths


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, json, re, random, time, shutil
from collections import Counter
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/tow_project/B/extension/scicite")
DATA_DIR = PROJECT_ROOT / "data"

# Clean Inline Word-Level ToW output directory.
# The directory name is kept for compatibility with the evaluation notebooks.
WORDLEVEL_CLEAN_DIR = PROJECT_ROOT / "wordlevel_tow_generations_clean_v2"
IW_TOW_DIR = WORDLEVEL_CLEAN_DIR

for p in [DATA_DIR, WORDLEVEL_CLEAN_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("IW_TOW_DIR:", IW_TOW_DIR)


## 3. Generation configuration


In [ ]:
OPENAI_MODEL = "gpt-5.4-mini"
OPENAI_CHECKER_MODEL = OPENAI_MODEL

GENERATE_SPLITS = ["train", "validation", "test"]
# Use e.g. {"train": 5, "validation": 5, "test": 5} for a smoke test.
GENERATION_LIMITS = {"train": None, "validation": None, "test": None}

SEED = 42
MAX_RETRIES = 4
SLEEP_BASE_SECONDS = 2
API_THROTTLE_SECONDS = 0.0
TARGET_SELECTION_VERSION = "v2_strict_contentword_no_citation_artifacts"

random.seed(SEED)
print("OPENAI_MODEL:", OPENAI_MODEL)
print("TARGET_SELECTION_VERSION:", TARGET_SELECTION_VERSION)


## 4. JSONL helpers


In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def append_jsonl(row, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def write_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def extract_json_object(text):
    if not isinstance(text, str):
        raise ValueError("Expected string response")
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        raise ValueError(f"No JSON object found in response: {text[:500]}")
    return json.loads(m.group(0))

print("Helpers loaded.")

## 5. Verify raw SciCite inputs


In [ ]:
expected = {
    "train": DATA_DIR / "scicite_train_raw_sample.jsonl",
    "validation": DATA_DIR / "scicite_validation_raw_sample.jsonl",
    "test": DATA_DIR / "scicite_test_raw_sample.jsonl",
}
for split, path in expected.items():
    rows = read_jsonl(path)
    print(split, path, "rows=", len(rows))
    if not rows:
        raise RuntimeError(
            f"Missing or empty raw sample file for {split}: {path}\n"
            "Run the SciCite raw-data/sample cell from the earlier notebook first."
        )

print("\nExample raw row:")
print(json.dumps(read_jsonl(expected["train"])[0], indent=2, ensure_ascii=False)[:1500])

## 6. Target-word selection

The target selector rejects citation artifacts such as author names, years, parenthetical citation fragments, short tokens, stopwords, and capitalized proper nouns. This keeps IW-ToW focused on semantic content words rather than citation metadata.


In [ ]:
STOPWORDS = {
    "the","a","an","and","or","but","if","then","else","for","to","from","of","in","on","at","by","with",
    "without","within","between","into","onto","over","under","through","during","before","after","as",
    "is","are","was","were","be","been","being","am","do","does","did","doing","done","have","has","had",
    "having","can","could","may","might","must","shall","should","will","would","this","that","these",
    "those","it","its","they","them","their","there","here","we","our","ours","you","your","i","he","she",
    "his","her","hers","which","who","whom","whose","what","when","where","why","how","not","no","yes",
    "than","so","such","also","very","more","most","less","least","only","just","both","each","all","any",
    "some","many","much","other","another","same","using","used","use","based","according"
}
BAD_TARGET_WORDS = {
    "et","al","fig","figure","table","supplementary","appendix","citation","citations","reference",
    "references","paper","papers","author","authors","study","studies","article","articles","section",
    "equation","eq","see","shown","described","roshan","fisher"
}
BAD_TOW_PHRASES = [
    "cited study","another cited","citation","citations","author","authors","et al","paper by",
    "reference","references","bibliographic","parenthetical","names a study","introduces another cited",
    "cited work"
]
WORD_RE = re.compile(r"\b[A-Za-z][A-Za-z\-]{3,}\b")

def inside_brackets_or_parentheses(text, idx):
    before = text[:idx]
    last_open = max(before.rfind("("), before.rfind("["))
    last_close = max(before.rfind(")"), before.rfind("]"))
    return last_open > last_close

def has_enough_prefix_words(prefix, min_words=8):
    return len(re.findall(r"\b\w+\b", prefix)) >= min_words

def near_citation_artifact(text, start, end):
    before = text[max(0, start-60):start]
    after = text[end:end+80]
    window = (before + " " + after).lower()
    if " et al" in after.lower()[:35] or " et al" in before.lower()[-35:]:
        return True
    if re.search(r"\b(19|20)\d{2}[a-z]?\b", window):
        return True
    nearby = text[max(0, start-30):min(len(text), end+80)].lower()
    if ("et al" in nearby or re.search(r"\b(19|20)\d{2}[a-z]?\b", nearby)) and re.search(r"\([^)]+\)", nearby):
        return True
    return False

def looks_like_bad_target(word, text, start, end):
    w = word.strip()
    wl = w.lower()
    if not w or wl in STOPWORDS or wl in BAD_TARGET_WORDS:
        return True
    if len(wl) < 4:
        return True
    if not re.fullmatch(r"[A-Za-z][A-Za-z\-]+", w):
        return True
    if re.fullmatch(r"\d+(\.\d+)?", w) or re.fullmatch(r"(19|20)\d{2}", w):
        return True
    # Reject likely proper nouns / author names. This also rejects sentence-start words, but gives cleaner ToW.
    if w[0].isupper():
        return True
    if inside_brackets_or_parentheses(text, start):
        return True
    if near_citation_artifact(text, start, end):
        return True
    if not has_enough_prefix_words(text[:start].strip(), min_words=8):
        return True
    return False

def get_candidate_targets(text):
    candidates = []
    for m in WORD_RE.finditer(text):
        word = m.group(0)
        start, end = m.start(), m.end()
        if not looks_like_bad_target(word, text, start, end):
            candidates.append({"word": word, "start": start, "end": end, "prefix": text[:start].strip()})
    return candidates

def choose_target(row):
    candidates = get_candidate_targets(row["text"])
    if not candidates:
        return None
    rng = random.Random(f"{SEED}-{row.get('id', '')}-{TARGET_SELECTION_VERSION}")
    return rng.choice(candidates)

preview = []
for split in GENERATE_SPLITS:
    for row in read_jsonl(expected[split])[:10]:
        target = choose_target(row)
        preview.append({
            "split": split,
            "id": row["id"],
            "target": target["word"] if target else None,
            "prefix_tail": target["prefix"][-120:] if target else None,
            "num_candidates": len(get_candidate_targets(row["text"])),
        })
pd.DataFrame(preview)

## 7. OpenAI API setup and prompts

The notebook first checks Colab Secrets for `OPENAI_API_KEY`. If it is not found, it prompts for the key using hidden input.


In [ ]:
from getpass import getpass
from openai import OpenAI

try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
except Exception:
    pass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

client = OpenAI()

GENERATOR_SYSTEM_PROMPT = """
You generate Thoughts of Words (ToW) annotations for next-word prediction.
You are shown only the text before a hidden target word.
Predict the likely next word and give a short reason based only on the prefix.
Do not discuss citation labels, SciCite, background/method/result classes, or paper intent.
Return JSON only.
""".strip()

CHECKER_SYSTEM_PROMPT = """
You check and denoise Thoughts of Words annotations.
You are given the prefix, the observed gold next word, the predicted next word, and the raw thought.
Classify the prediction as exact_match, soft_consistent, or unpredictable.
Then write a denoised ToW under 15 words explaining why the observed word follows from the prefix.
Avoid citation-role language and avoid mentioning authors/citations unless unavoidable.
Return JSON only.
""".strip()

def _response_text(resp):
    if hasattr(resp, "output_text") and resp.output_text:
        return resp.output_text
    return str(resp)

def responses_json(model, system_prompt, user_prompt, max_output_tokens=300, include_temperature=False):
    kwargs = {
        "model": model,
        "input": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        "max_output_tokens": max_output_tokens,
    }
    if include_temperature:
        kwargs["temperature"] = 0.2
    resp = client.responses.create(**kwargs)
    return extract_json_object(_response_text(resp))

def chat_json(model, system_prompt, user_prompt, max_tokens=300):
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        response_format={"type": "json_object"},
        max_tokens=max_tokens,
    )
    return extract_json_object(resp.choices[0].message.content)

def call_json_with_retries(model, system_prompt, user_prompt, max_output_tokens=300):
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        for use_temp in [False, True]:
            try:
                return responses_json(model, system_prompt, user_prompt, max_output_tokens=max_output_tokens, include_temperature=use_temp)
            except Exception as e:
                last_err = e
        try:
            return chat_json(model, system_prompt, user_prompt, max_tokens=max_output_tokens)
        except Exception as e:
            last_err = e
            print(f"Attempt {attempt}/{MAX_RETRIES} failed: {repr(last_err)}")
            time.sleep(SLEEP_BASE_SECONDS * attempt)
    raise RuntimeError(f"API failed after {MAX_RETRIES} attempts. Last error: {repr(last_err)}")

print("OpenAI client and API helpers ready.")


## 8. IW-ToW generation and validation functions


In [ ]:
def clean_prefix_for_prompt(prefix, max_chars=2200):
    prefix = prefix.strip()
    return prefix if len(prefix) <= max_chars else prefix[-max_chars:]

def generate_word_thought(prefix, model=OPENAI_MODEL):
    prompt = f"""
Prefix before the hidden next word:
{clean_prefix_for_prompt(prefix)}

Return a JSON object with exactly these keys:
- predicted_next_word: one likely next word, lower-case if possible
- thought: one sentence, under 25 words, explaining why that word follows from the prefix

Important:
- You do not know the true next word.
- Do not talk about citation intent or SciCite labels.
- Do not mention "background", "method", or "result".
""".strip()
    data = call_json_with_retries(model, GENERATOR_SYSTEM_PROMPT, prompt, max_output_tokens=220)
    return {"predicted_next_word": str(data.get("predicted_next_word", "")).strip(), "thought": str(data.get("thought", "")).strip()}

def check_and_denoise(prefix, target_word, predicted_next_word, raw_thought, model=OPENAI_CHECKER_MODEL):
    prompt = f"""
Prefix:
{clean_prefix_for_prompt(prefix)}

Observed gold next word:
{target_word}

Predicted next word:
{predicted_next_word}

Raw thought:
{raw_thought}

Return a JSON object with exactly these keys:
- category: one of exact_match, soft_consistent, unpredictable
- denoised_tow: under 15 words; explain why the observed gold word follows from the prefix
- keep: true if category is exact_match or soft_consistent and the thought is useful, otherwise false

Rules:
- exact_match means predicted_next_word is the same word as the observed gold word, ignoring case/simple punctuation.
- soft_consistent means the prediction is not exact but semantically compatible with the gold word.
- unpredictable means the gold word does not reasonably follow from the prefix.
- denoised_tow should not become a citation-role explanation.
- Avoid phrases like "cited study", "author", "citation", "et al.", "background", "method", or "result".
""".strip()
    data = call_json_with_retries(model, CHECKER_SYSTEM_PROMPT, prompt, max_output_tokens=260)
    category = str(data.get("category", "")).strip()
    if category not in {"exact_match", "soft_consistent", "unpredictable"}:
        category = "unpredictable"
    keep = bool(data.get("keep", category in {"exact_match", "soft_consistent"}))
    return {"category": category, "denoised_tow": str(data.get("denoised_tow", "")).strip(), "keep": keep}

def insert_tow(text, target_start, tow):
    return text[:target_start] + f"<ToW> {tow.strip()} </ToW> " + text[target_start:]

def validate_clean_wordlevel_row(row):
    if row.get("target_selection_version") != TARGET_SELECTION_VERSION:
        return False
    if not isinstance(row.get("id"), str):
        return False
    if not isinstance(row.get("text"), str) or not isinstance(row.get("tow_augmented_text"), str):
        return False
    if "<ToW>" not in row["tow_augmented_text"] or "</ToW>" not in row["tow_augmented_text"]:
        return False
    if row.get("tow_category") not in {"exact_match", "soft_consistent", "unpredictable"}:
        return False
    if row.get("tow_category") == "unpredictable":
        return False
    target = row.get("target_word", "")
    if not target or looks_like_bad_target(target, row["text"], int(row.get("target_start", 0)), int(row.get("target_end", 0))):
        return False
    tow = (row.get("wordlevel_tow") or "").lower()
    if len(tow.split()) > 18:
        return False
    if any(p in tow for p in BAD_TOW_PHRASES):
        return False
    return True

print("Generation/checker functions ready.")

## 9. Generate IW-ToW files

Generation is restart-safe. If a valid row already exists for an ID, it is skipped. Rejected and errored examples are written to separate JSONL files for auditing.


In [ ]:
def generate_clean_wordlevel_split(split, limit=None):
    raw_path = DATA_DIR / f"scicite_{split}_raw_sample.jsonl"
    out_path = WORDLEVEL_CLEAN_DIR / f"scicite_{split}_wordlevel_tow.jsonl"
    progress_path = WORDLEVEL_CLEAN_DIR / f"scicite_{split}_wordlevel_tow_progress.json"

    raw_rows = read_jsonl(raw_path)
    existing = read_jsonl(out_path)
    existing_valid = {r["id"]: r for r in existing if validate_clean_wordlevel_row(r)}

    print(f"\n=== {split} ===")
    print("raw rows:", len(raw_rows))
    print("valid cached IW-ToW rows:", len(existing_valid))
    print("out:", out_path)

    todo = [r for r in raw_rows if r["id"] not in existing_valid]
    if limit is not None:
        todo = todo[:limit]
    print("to generate:", len(todo))

    started = time.time()
    skipped_no_target = 0
    skipped_bad_checker = 0

    for i, row in enumerate(todo, 1):
        target = choose_target(row)
        if target is None:
            skipped_no_target += 1
            continue

        try:
            gen = generate_word_thought(target["prefix"])
            check = check_and_denoise(target["prefix"], target["word"], gen["predicted_next_word"], gen["thought"])

            base = dict(row)
            base.update({
                "openai_model": OPENAI_MODEL,
                "checker_model": OPENAI_CHECKER_MODEL,
                "target_word": target["word"],
                "target_start": target["start"],
                "target_end": target["end"],
                "prefix": target["prefix"],
                "predicted_next_word": gen["predicted_next_word"],
                "raw_word_thought": gen["thought"],
                "tow_category": check["category"],
                "wordlevel_tow": check["denoised_tow"],
                "tow_augmented_text": insert_tow(row["text"], target["start"], check["denoised_tow"]),
                "wordlevel_checker_used": True,
                "target_selection_version": TARGET_SELECTION_VERSION,
                "generated_at_unix": time.time(),
            })

            if check["category"] not in {"exact_match", "soft_consistent"} or not check["keep"]:
                skipped_bad_checker += 1
                base["rejected_reason"] = "checker_rejected"
                append_jsonl(base, WORDLEVEL_CLEAN_DIR / f"scicite_{split}_wordlevel_tow_rejected.jsonl")
                continue

            if not validate_clean_wordlevel_row(base):
                skipped_bad_checker += 1
                base["rejected_reason"] = "local_validation_failed"
                append_jsonl(base, WORDLEVEL_CLEAN_DIR / f"scicite_{split}_wordlevel_tow_rejected.jsonl")
                continue

            append_jsonl(base, out_path)

            if i <= 3 or i % 10 == 0 or i == len(todo):
                print(f"[{split}] {i}/{len(todo)} id={row['id']} target={target['word']} pred={gen['predicted_next_word']} cat={check['category']} tow={check['denoised_tow'][:90]}")

            if API_THROTTLE_SECONDS:
                time.sleep(API_THROTTLE_SECONDS)

        except Exception as e:
            append_jsonl({"id": row.get("id"), "split": split, "error": repr(e), "target": target, "time": time.time()}, WORDLEVEL_CLEAN_DIR / f"scicite_{split}_wordlevel_tow_errors.jsonl")
            print(f"[{split}] ERROR id={row.get('id')}: {repr(e)}")
            time.sleep(SLEEP_BASE_SECONDS)

        current_valid = [r for r in read_jsonl(out_path) if validate_clean_wordlevel_row(r)]
        write_json({
            "split": split,
            "raw_total": len(raw_rows),
            "valid_iw_tow_rows": len(current_valid),
            "skipped_no_target_this_run": skipped_no_target,
            "skipped_bad_checker_this_run": skipped_bad_checker,
            "model": OPENAI_MODEL,
            "checker_model": OPENAI_CHECKER_MODEL,
            "target_selection_version": TARGET_SELECTION_VERSION,
            "elapsed_seconds": round(time.time() - started, 2),
            "path": str(out_path),
        }, progress_path)

    final_rows = [r for r in read_jsonl(out_path) if validate_clean_wordlevel_row(r)]
    print(f"Done {split}: valid IW-ToW rows={len(final_rows)} / raw={len(raw_rows)}")
    return out_path

for split in GENERATE_SPLITS:
    generate_clean_wordlevel_split(split, GENERATION_LIMITS.get(split))


## 10. Summarize generated IW-ToW data


In [ ]:
summary = []
for split in GENERATE_SPLITS:
    path = WORDLEVEL_CLEAN_DIR / f"scicite_{split}_wordlevel_tow.jsonl"
    rows = read_jsonl(path)
    valid = [r for r in rows if validate_clean_wordlevel_row(r)]
    cats = Counter(r.get("tow_category") for r in valid)
    labels = Counter(r.get("label_name") for r in valid)
    targets = Counter(r.get("target_word") for r in valid)
    summary.append({
        "split": split,
        "rows": len(rows),
        "valid_iw_tow": len(valid),
        "exact_match": cats.get("exact_match", 0),
        "soft_consistent": cats.get("soft_consistent", 0),
        "unique_targets": len(targets),
        "top_targets": targets.most_common(10),
        "labels": dict(labels),
        "path": str(path),
    })

summary_path = WORDLEVEL_CLEAN_DIR / "inline_wordlevel_tow_summary.json"
write_json(summary, summary_path)

df = pd.DataFrame(summary)
display(df[["split", "rows", "valid_iw_tow", "exact_match", "soft_consistent", "unique_targets", "path"]])

print("\nExamples:")
for split in GENERATE_SPLITS:
    valid = [r for r in read_jsonl(WORDLEVEL_CLEAN_DIR / f"scicite_{split}_wordlevel_tow.jsonl") if validate_clean_wordlevel_row(r)]
    print("\n---", split, "---")
    for r in valid[:3]:
        print("id:", r["id"], "| target:", r["target_word"], "| category:", r["tow_category"])
        print(r["tow_augmented_text"][:600])
        print()


## 11. Qualitative inspection


In [ ]:
for split in GENERATE_SPLITS:
    valid = [
        r for r in read_jsonl(WORDLEVEL_CLEAN_DIR / f"scicite_{split}_wordlevel_tow.jsonl")
        if validate_clean_wordlevel_row(r)
    ]
    print("\n---", split, "---")
    for r in valid[:3]:
        print("id:", r["id"], "| target:", r["target_word"], "| category:", r["tow_category"])
        print(r["tow_augmented_text"][:700])
        print()

## 12. Downstream path

Use the path printed below in the direct-input and ToW-MLM Adaptation notebooks.


In [ ]:
print("Inline Word-Level ToW files:")
for split in GENERATE_SPLITS:
    p = WORDLEVEL_CLEAN_DIR / f"scicite_{split}_wordlevel_tow.jsonl"
    print(split, p, "rows=", len(read_jsonl(p)))

print("\nUse this path in the direct-input and ToW-MLM Adaptation notebooks:")
print(f'IW_TOW_DIR = Path("{WORDLEVEL_CLEAN_DIR}")')
